In [2]:
import pandas as pd
import numpy as np

model_cols = ['timestamp', 'is_skip', 'session_position', 'session_progress',
              'prev_is_skip', 'rolling_skip_rate_5', 'artist_repeat', 'genre_changed',
              'hour_sin', 'hour_cos', 'day_of_week', 'user_historical_skip_rate',
              'energy_delta', 'valence_delta', 'danceability_delta',
              'rolling_energy_5', 'rolling_valence_5']

df_model = pd.read_parquet('../data/processed/feature_engineered.parquet', columns=model_cols)

float_cols_to_downcast = ['session_progress', 'rolling_skip_rate_5', 'hour_sin', 'hour_cos',
                           'user_historical_skip_rate', 'energy_delta', 'valence_delta',
                           'danceability_delta', 'rolling_energy_5', 'rolling_valence_5']
for col in float_cols_to_downcast:
    df_model[col] = df_model[col].astype('float32')

is_outlier = (df_model['timestamp'].dt.year >= 2010)
cutoff = df_model.loc[~is_outlier, 'timestamp'].quantile(0.8)

train = df_model[(~is_outlier) & (df_model['timestamp'] < cutoff)]
test = df_model[(~is_outlier) & (df_model['timestamp'] >= cutoff)]

behavioral_features = [
    'session_position', 'session_progress', 'prev_is_skip', 'rolling_skip_rate_5',
    'artist_repeat', 'genre_changed', 'hour_sin', 'hour_cos', 'day_of_week',
    'user_historical_skip_rate'
]

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Train skip rate: {train['is_skip'].mean():.4f}, Test skip rate: {test['is_skip'].mean():.4f}")

Train: (15320692, 17), Test: (3830174, 17)
Train skip rate: 0.0065, Test skip rate: 0.0148


In [17]:
is_outlier = (df_model['timestamp'].dt.year >= 2010)  # simpler outlier check, avoids the extra year_month column
print(f"Outlier rows found: {is_outlier.sum()}")

cutoff = df_model.loc[~is_outlier, 'timestamp'].quantile(0.8)

# No .copy() — we're not modifying train/test in place, just reading from them,
# so a view/reference is enough and avoids duplicating memory
train = df_model[(~is_outlier) & (df_model['timestamp'] < cutoff)]
test = df_model[(~is_outlier) & (df_model['timestamp'] >= cutoff)]

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Cutoff date: {cutoff}")
print(f"Train skip rate: {train['is_skip'].mean():.4f}, Test skip rate: {test['is_skip'].mean():.4f}")

Outlier rows found: 2
Train: (15320692, 17), Test: (3830174, 17)
Cutoff date: 2008-10-16 19:54:49+00:00
Train skip rate: 0.0065, Test skip rate: 0.0148


In [9]:
behavioral_features = [
    'session_position', 'session_progress', 'prev_is_skip', 'rolling_skip_rate_5',
    'artist_repeat', 'genre_changed', 'hour_sin', 'hour_cos', 'day_of_week',
    'user_historical_skip_rate'
]

X_train_a = train[behavioral_features]
y_train_a = train['is_skip']
X_test_a = test[behavioral_features]
y_test_a = test['is_skip']

print(X_train_a.shape, X_test_a.shape)
print(X_train_a.isna().sum())  # should be all zeros — these are the fully-populated features

(15320692, 10) (3830174, 10)
session_position             0
session_progress             0
prev_is_skip                 0
rolling_skip_rate_5          0
artist_repeat                0
genre_changed                0
hour_sin                     0
hour_cos                     0
day_of_week                  0
user_historical_skip_rate    0
dtype: int64


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Scale features — logistic regression is sensitive to feature scale,
# and our features range wildly (day_of_week is 0-6, hour_sin is -1 to 1, session_position can be in the hundreds)
scaler = StandardScaler()
X_train_a_scaled = scaler.fit_transform(X_train_a)
X_test_a_scaled = scaler.transform(X_test_a)

# class_weight='balanced' compensates for the ~0.8% skip rate — without this,
# the model could get ~99% accuracy by just always predicting "not skipped," which is useless
model_a = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model_a.fit(X_train_a_scaled, y_train_a)

# Predict probabilities, not just hard labels — needed for AUC and lets us tune the decision threshold later
y_pred_proba_a = model_a.predict_proba(X_test_a_scaled)[:, 1]
y_pred_a = model_a.predict(X_test_a_scaled)

print("AUC:", roc_auc_score(y_test_a, y_pred_proba_a))
print("\nClassification report:")
print(classification_report(y_test_a, y_pred_a))
print("\nConfusion matrix:")
print(confusion_matrix(y_test_a, y_pred_a))

AUC: 0.9213262626445122

Classification report:
              precision    recall  f1-score   support

       False       1.00      0.93      0.96   3773620
        True       0.15      0.84      0.26     56554

    accuracy                           0.93   3830174
   macro avg       0.57      0.89      0.61   3830174
weighted avg       0.98      0.93      0.95   3830174


Confusion matrix:
[[3504400  269220]
 [   8884   47670]]


In [21]:
coef_df = pd.DataFrame({
    'feature': behavioral_features,
    'coefficient': model_a.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df)

                     feature  coefficient
3        rolling_skip_rate_5     2.080132
9  user_historical_skip_rate     0.923893
2               prev_is_skip    -0.917344
4              artist_repeat    -0.387337
1           session_progress    -0.292843
0           session_position     0.173677
7                   hour_cos     0.031240
8                day_of_week    -0.023949
5              genre_changed     0.023402
6                   hour_sin    -0.003982


In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.3, 0.5, 0.7, 0.8, 0.9]
results = []

for t in thresholds:
    y_pred_t = (y_pred_proba_a >= t).astype(int)
    results.append({
        'threshold': t,
        'precision': precision_score(y_test_a, y_pred_t),
        'recall': recall_score(y_test_a, y_pred_t),
        'f1': f1_score(y_test_a, y_pred_t),
        'n_flagged': y_pred_t.sum()
    })

threshold_df = pd.DataFrame(results)
print(threshold_df)

NameError: name 'y_pred_proba_a' is not defined

In [3]:
audio_features = ['energy_delta', 'valence_delta', 'danceability_delta',
                   'rolling_energy_5', 'rolling_valence_5']

all_features_b = behavioral_features + audio_features

# Select only the needed columns first (small), THEN fill — no full train/test copy needed
X_train_b = train[all_features_b].copy()
X_test_b = test[all_features_b].copy()

for col in ['energy_delta', 'valence_delta', 'danceability_delta']:
    X_train_b[col] = X_train_b[col].fillna(-1)
    X_test_b[col] = X_test_b[col].fillna(-1)

y_train_b = train['is_skip']
y_test_b = test['is_skip']

print(X_train_b.isna().sum())

session_position             0
session_progress             0
prev_is_skip                 0
rolling_skip_rate_5          0
artist_repeat                0
genre_changed                0
hour_sin                     0
hour_cos                     0
day_of_week                  0
user_historical_skip_rate    0
energy_delta                 0
valence_delta                0
danceability_delta           0
rolling_energy_5             0
rolling_valence_5            0
dtype: int64


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

In [6]:
scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

model_b = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model_b.fit(X_train_b_scaled, y_train_b)

y_pred_proba_b = model_b.predict_proba(X_test_b_scaled)[:, 1]
y_pred_b = model_b.predict(X_test_b_scaled)

print("Model B AUC:", roc_auc_score(y_test_b, y_pred_proba_b))
print("\nClassification report:")
print(classification_report(y_test_b, y_pred_b))

Model B AUC: 0.9215372004906731

Classification report:
              precision    recall  f1-score   support

       False       1.00      0.93      0.96   3773620
        True       0.15      0.84      0.25     56554

    accuracy                           0.93   3830174
   macro avg       0.57      0.89      0.61   3830174
weighted avg       0.98      0.93      0.95   3830174



In [7]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.6/101.7 MB 9.4 MB/s eta 0:00:11
   - -------------------------------------- 3.9/101.7 MB 10.2 MB/s eta 0:00:10
   -- ------------------------------------- 6.3/101.7 MB 10.7 MB/s eta 0:00:09
   --- ------------------------------------ 8.7/101.7 MB 10.5 MB/s eta 0:00:09
   ---- ----------------------------------- 11.0/101.7 MB 10.7 MB/s eta 0:00:09
   ----- ---------------------------------- 13.6/101.7 MB 11.0 MB/s eta 0:00:09
   ------ --------------------------------- 16.5/101.7 MB 11.3 MB/s eta 0:00:08
   ------- -------------------------------- 19.1/101.7 MB 11.5 MB/s eta 0:00:08
   -------- ------------------------------- 21.8/101.7 MB 11.5 MB/s eta 0:00:07
   --------- ------------------------------ 24.1/101.7 MB 11.4 MB/s eta 0:00:07
   ---------- ----------------------------- 26.5/101.7 MB 11.4 MB/s eta 0:00:07
   ----------- ---------------------------- 29.4/101.7


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

# scale_pos_weight compensates for class imbalance, XGBoost's equivalent of class_weight='balanced'
neg, pos = (y_train_a == 0).sum(), (y_train_a == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

model_c = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

model_c.fit(X_train_a, y_train_a)  # no scaling needed for tree models

y_pred_proba_c = model_c.predict_proba(X_test_a)[:, 1]
y_pred_c = model_c.predict(X_test_a)

print("XGBoost AUC:", roc_auc_score(y_test_a, y_pred_proba_c))
print("\nClassification report:")
print(classification_report(y_test_a, y_pred_c))

scale_pos_weight: 154.0
XGBoost AUC: 0.9618273463893122

Classification report:
              precision    recall  f1-score   support

       False       1.00      0.86      0.93   3773620
        True       0.09      0.91      0.16     56554

    accuracy                           0.86   3830174
   macro avg       0.54      0.89      0.55   3830174
weighted avg       0.98      0.86      0.91   3830174



In [14]:
thresholds = [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]
results_c = []

for t in thresholds:
    y_pred_t = (y_pred_proba_c >= t).astype(int)
    results_c.append({
        'threshold': t,
        'precision': precision_score(y_test_a, y_pred_t),
        'recall': recall_score(y_test_a, y_pred_t),
        'f1': f1_score(y_test_a, y_pred_t)
    })

threshold_df_c = pd.DataFrame(results_c)
print(threshold_df_c)

   threshold  precision    recall        f1
0       0.30   0.048839  0.959702  0.092947
1       0.50   0.090430  0.907593  0.164473
2       0.70   0.186343  0.848463  0.305574
3       0.80   0.280370  0.821304  0.418034
4       0.90   0.412769  0.782650  0.540486
5       0.95   0.520065  0.738781  0.610423


In [15]:
importance_df = pd.DataFrame({
    'feature': behavioral_features,
    'importance': model_c.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df)

                     feature  importance
3        rolling_skip_rate_5    0.922303
9  user_historical_skip_rate    0.048975
2               prev_is_skip    0.005545
1           session_progress    0.005437
4              artist_repeat    0.005085
0           session_position    0.004368
8                day_of_week    0.002599
5              genre_changed    0.002067
6                   hour_sin    0.001932
7                   hour_cos    0.001688


In [16]:
import os
os.makedirs('../models', exist_ok=True)

model_c.save_model('../models/xgboost_skip_model.json')
print("Model saved")

# Also save a manageable sample of the test set for SHAP analysis —
# SHAP is computationally expensive, and running it on all 3.8M test rows
# would be extremely slow (and memory-heavy) for little added insight.
# A random sample of ~50,000 rows gives stable, representative SHAP values.
X_test_a_sample = X_test_a.sample(n=50_000, random_state=42)
X_test_a_sample.to_parquet('../data/processed/shap_sample.parquet', index=False)
print("SHAP sample saved:", X_test_a_sample.shape)

Model saved
SHAP sample saved: (50000, 10)
